In [1]:
import requests
import pandas as pd
import datetime as dt
import time
import os

BASE_URL = "https://api.openaq.org/v3/locations"
HEADERS = {"X-API-Key": "b7e76c96871664141e9a369a692613768c5fcdc7615e7249ccf7463bd03bc7b3"}  # store API key in environment

country_id = 155
country_name = "UnitedStates"

def get_all_locations():
    all_results = []
    page = 1

    while True:
        print(f"Fetching page {page}")
        params = {
            "countries_id": country_id,
            "limit": 100,
            "page": page,
            "order_by": "id",
            "sort_order": "desc"
        }

        response = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=30)

        # Rate-limit handling
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 30))
            print(f"Rate limited. Sleeping {retry_after} seconds...")
            time.sleep(retry_after)
            continue

        if response.status_code != 200:
            print(f"Failed with status {response.status_code}")
            print(response.text)
            break

        data = response.json()
        results = data.get("results", [])

        if not results:
            print("Extraction complete.")
            break

        all_results.extend(results)
        page += 1

    return all_results

def build_locations_raw(raw_results):
    df = pd.json_normalize(raw_results)

    df_clean = df[[
        "id",
        "name",
        "locality",           
        "country.code",
        "coordinates.latitude",
        "coordinates.longitude",
        "sensors",
        "timezone",
        "isMobile",
        "datetimeFirst.utc",
        "datetimeLast.utc"
    ]].copy()

    df_clean.columns = [
        "location_id",
        "name",
        "locality",
        "country",
        "latitude",
        "longitude",
        "sensors",
        "timezone",
        "is_mobile",
        "first_updated_utc",
        "last_updated_utc"
    ]

    return df_clean

# Columns to keep
SENSOR_COLUMNS = [
    'id', 'name', 'parameter.id', 'parameter.name', 'parameter.units',
    'datetimeFirst.utc', 'datetimeLast.utc', 'latest.value',
    'summary.min', 'summary.q02', 'summary.q25', 'summary.median',
    'summary.q75', 'summary.q98', 'summary.max', 'summary.avg', 'summary.sd'
]

def fetch_sensor_metadata(sensor_id, max_retries=3):
    """Fetch metadata for a single sensor with safe retry handling."""
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}"
    retries = 0

    while retries < max_retries:
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)

            if r.status_code == 429:
                retry_after = int(r.headers.get("Retry-After", 10))
                print(f"Rate limited for sensor {sensor_id}. Sleeping {retry_after} seconds...")
                time.sleep(retry_after)
                retries += 1
                continue

            elif r.status_code == 200:
                return r.json().get("results", [])

            else:
                print(f"Failed sensor {sensor_id}: {r.status_code}")
                return None

        except requests.exceptions.RequestException as e:
            print(f"Exception for sensor {sensor_id}: {e}")
            retries += 1
            time.sleep(5)

    print(f"Max retries exceeded for sensor {sensor_id}")
    return None

In [2]:
# -----------------------------
# Main execution
# -----------------------------

if __name__ == "__main__":

    # Locations
    raw_locations = get_all_locations()



Fetching page 1
Fetching page 2
Fetching page 3
Fetching page 4
Fetching page 5
Fetching page 6
Fetching page 7
Fetching page 8
Fetching page 9
Fetching page 10
Fetching page 11
Fetching page 12
Fetching page 13
Fetching page 14
Fetching page 15
Fetching page 16
Fetching page 17
Fetching page 18
Fetching page 19
Fetching page 20
Fetching page 21
Fetching page 22
Fetching page 23
Fetching page 24
Fetching page 25
Fetching page 26
Fetching page 27
Fetching page 28
Fetching page 29
Fetching page 30
Fetching page 31
Fetching page 32
Fetching page 33
Fetching page 34
Fetching page 35
Fetching page 36
Fetching page 37
Fetching page 38
Fetching page 39
Fetching page 40
Fetching page 41
Fetching page 42
Fetching page 43
Fetching page 44
Fetching page 45
Fetching page 46
Fetching page 47
Fetching page 48
Fetching page 49
Fetching page 50
Fetching page 51
Fetching page 52
Fetching page 53
Fetching page 54
Fetching page 55
Fetching page 56
Fetching page 57
Fetching page 58
Fetching page 59
Fetchi

In [3]:
def fetch_sensor_metadata(sensor_id, max_retries=3):
    """Fetch metadata for a single sensor with safe retry handling."""
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}"
    retries = 0

    while retries < max_retries:
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)

            if r.status_code == 429:
                retry_after = int(r.headers.get("Retry-After", 10))
                print(f"Rate limited for sensor {sensor_id}. Sleeping {retry_after} seconds...")
                time.sleep(retry_after)
                retries += 1
                continue

            elif r.status_code == 200:
                return r.json().get("results", [])

            else:
                print(f"Failed sensor {sensor_id}: {r.status_code}")
                return None

        except requests.exceptions.RequestException as e:
            print(f"Exception for sensor {sensor_id}: {e}")
            retries += 1
            time.sleep(5)

    print(f"Max retries exceeded for sensor {sensor_id}")
    return None

data = []

for loc in raw_locations:
    for sensor in loc["sensors"]:
        data.append({
        "location_id": loc["id"],
        "sensor_id": sensor["id"]
        })

results = []

for element in data:
    loc_id = element["location_id"]
    sensor_id = element["sensor_id"]

    metadata = fetch_sensor_metadata(sensor_id=sensor_id)

    if metadata:
        for item in metadata:  # because API returns a list in "results"
            item["location_id"] = loc_id
            item["sensor_id"] = sensor_id
            print(item)
            results.append(item)

{'id': 15582149, 'name': 'pm1 µg/m³', 'parameter': {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': 'PM1'}, 'datetimeFirst': {'utc': '2026-03-03T02:00:00Z', 'local': '2026-03-02T20:00:00-06:00'}, 'datetimeLast': {'utc': '2026-03-03T09:00:00Z', 'local': '2026-03-03T03:00:00-06:00'}, 'coverage': {'expectedCount': 1, 'expectedInterval': '01:00:00', 'observedCount': 8, 'observedInterval': '08:00:00', 'percentComplete': 800.0, 'percentCoverage': 800.0, 'datetimeFrom': {'utc': '2026-03-03T02:00:00Z', 'local': '2026-03-02T20:00:00-06:00'}, 'datetimeTo': {'utc': '2026-03-03T09:00:00Z', 'local': '2026-03-03T03:00:00-06:00'}}, 'latest': {'datetime': {'utc': '2026-03-03T09:00:00Z', 'local': '2026-03-03T03:00:00-06:00'}, 'value': 0.0, 'coordinates': {'latitude': 43.05643, 'longitude': -88.133679}}, 'summary': {'min': 0.0, 'q02': None, 'q25': None, 'median': None, 'q75': None, 'q98': None, 'max': 0.0, 'avg': 0.0, 'sd': 0.0}, 'location_id': 6258339, 'sensor_id': 15582149}
{'id': 15582150, 

KeyboardInterrupt: 

{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 100, 'found': 3}, 'results': [{'value': 5.53, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': None}, 'period': {'label': '1day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2026-02-28T06:00:00Z', 'local': '2026-02-28T00:00:00-06:00'}, 'datetimeTo': {'utc': '2026-03-01T06:00:00Z', 'local': '2026-03-01T00:00:00-06:00'}}, 'coordinates': None, 'summary': {'min': 4.2, 'q02': 4.232, 'q25': 4.6, 'median': 5.0, 'q75': 6.2, 'q98': 7.304, 'max': 7.4, 'avg': 5.533333333333334, 'sd': None}, 'coverage': {'expectedCount': 24, 'expectedInterval': '24:00:00', 'observedCount': 3, 'observedInterval': '03:00:00', 'percentComplete': 13.0, 'percentCoverage': 13.0, 'datetimeFrom': {'utc': '2026-03-01T04:00:00Z', 'local': '2026-02-28T22:00:00-06:00'}, 'datetimeTo': {'utc': '2026-03-01T06:00:00Z', 'local': '2026-03-01T00:00:00-06:00'}}}, {'value': 9.6, 'flagInfo': {'hasFlags': Fals